In [ ]:
# Set up
from google.colab import drive
drive.mount('/content/drive')
!pip install mne --quiet
import os
import mne
import glob
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from scipy.fft import fft, ifft
from tqdm import tqdm
import scipy.io.wavfile as wav
from mne.time_frequency import tfr_array_morlet

In [ ]:
import mne
import numpy as np
import os

# ---- 1. Set path to your EEG data folder ----
data_folder = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

# ---- 2. List all participant files ----
all_files = [f for f in os.listdir(data_folder) if f.endswith('.set')]

# ---- 3. Create lists to store successful and failed participants ----
successful_files = []
failed_files = []

# ---- 4. Loop over participants ----
for idx, filename in enumerate(all_files, 1):
    print(f'[{idx}/{len(all_files)}] Processing file: {filename}', flush=True)

    try:
        # Load data from EEGLAB .set file
        raw = mne.io.read_raw_eeglab(os.path.join(data_folder, filename), preload=True)

        # Verify mastoid channels exist
        mastoid_channels = ['E56', 'E57', 'E100', 'E107']
        for ch in mastoid_channels:
            if ch not in raw.ch_names:
                raise ValueError(f'Channel {ch} not found in {filename}')

        # Extract mastoid signals
        lm1 = raw.copy().pick_channels(['E56']).get_data()[0]
        lm2 = raw.copy().pick_channels(['E57']).get_data()[0]
        rm1 = raw.copy().pick_channels(['E100']).get_data()[0]
        rm2 = raw.copy().pick_channels(['E107']).get_data()[0]

        # Compute linked mastoid
        linked_mastoid = (lm1 + lm2 + rm1 + rm2) / 4

        # Apply re-reference
        data = raw.get_data()
        data_reref = data - linked_mastoid
        raw._data = data_reref

        # Save as FIF file (MNE native format)
        output_file = os.path.join(data_folder, f'{filename[:-4]}_reref.fif')
        raw.save(output_file, overwrite=True)

        print(f'Finished re-referencing {filename}\n', flush=True)
        successful_files.append(filename)

    except Exception as e:
        print(f'Error processing {filename}: {e}', flush=True)
        failed_files.append(filename)
        continue

# ---- 5. Print summary after all participants ----
print("\n=== Summary ===")
print(f"Total files processed: {len(all_files)}")
print(f"Successful: {len(successful_files)}")
print(f"Failed: {len(failed_files)}")

if successful_files:
    print("\n✅ Successfully processed:")
    for s in successful_files:
        print(s)

if failed_files:
    print("\n❌ Failed to process:")
    for f in failed_files:
        print(f)



In [ ]:
import os

data_folder = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

# List all folders inside EEGData
subfolders = os.listdir(data_folder)
print("Subfolders:", subfolders)

# Let's list files inside one of them
for subfolder in subfolders:
    subfolder_path = os.path.join(data_folder, subfolder)
    if os.path.isdir(subfolder_path):
        files = os.listdir(subfolder_path)
        print(f"Files inside {subfolder}:", files)


In [ ]:
import mne
import numpy as np
import os
import glob

# 1️⃣ Mount Google Drive (only if you're in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# 2️⃣ Set correct EEG Data path
data_folder = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

# 3️⃣ List participants (folders with participant numbers)
participants = [f for f in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder, f))]
participants.sort()  # optional: sort numerically

print(f"Found {len(participants)} participants: {participants}")

# 4️⃣ Initialize success/fail lists for logging
successful_files = []
failed_files = []

# 5️⃣ Loop over each participant folder
for idx, participant in enumerate(participants, 1):
    print(f'\n[{idx}/{len(participants)}] Processing participant: {participant}', flush=True)

    participant_folder = os.path.join(data_folder, participant)

    # Find any .set file inside participant folder
    set_files = glob.glob(os.path.join(participant_folder, '*.set'))

    if not set_files:
        print(f'❌ No .set file found for participant {participant}')
        failed_files.append(participant)
        continue  # skip to next participant

    # Take the first .set file found
    filename = set_files[0]

    try:
        # Load EEGLAB .set file
        raw = mne.io.read_raw_eeglab(filename, preload=True)

        # Mastoid channels
        mastoid_channels = ['E56', 'E57', 'E100', 'E107']
        missing_channels = [ch for ch in mastoid_channels if ch not in raw.ch_names]

        if missing_channels:
            raise ValueError(f"Missing mastoid channels: {missing_channels}")

        # Extract mastoid signals
        lm1 = raw.copy().pick_channels(['E56']).get_data()[0]
        lm2 = raw.copy().pick_channels(['E57']).get_data()[0]
        rm1 = raw.copy().pick_channels(['E100']).get_data()[0]
        rm2 = raw.copy().pick_channels(['E107']).get_data()[0]

        # Compute linked mastoid average
        linked_mastoid = (lm1 + lm2 + rm1 + rm2) / 4

        # Apply re-reference
        data = raw.get_data()
        data_reref = data - linked_mastoid
        raw._data = data_reref

        # Save re-referenced file as FIF
        output_file = filename.replace('.set', '_reref.fif')
        raw.save(output_file, overwrite=True)

        print(f'✅ Finished re-referencing participant {participant}', flush=True)
        successful_files.append(participant)

    except Exception as e:
        print(f'❌ Error processing participant {participant}: {e}', flush=True)
        failed_files.append(participant)
        continue

# 6️⃣ Print final summary
print("\n=== Summary ===")
print(f"Total participants processed: {len(participants)}")
print(f"Successful: {len(successful_files)}")
print(f"Failed: {len(failed_files)}")

if successful_files:
    print("\n✅ Successfully processed:")
    for s in successful_files:
        print(s)

if failed_files:
    print("\n❌ Failed to process:")
    for f in failed_files:
        print(f)
